In [ ]:
from tst_sim_tools.agents.energy_alignment import build_qs_agent
from tst_sim_tools.analysis.image import analyze_image
from bluesky_queueserver_api.http import REManagerAPI
from bluesky_queueserver_api import BPlan, BItem
from tiled.client import from_uri
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
tiled_client = from_uri("http://127.0.0.1:36307/api/v1?api_key=3c610376bad5ec48")
RM = REManagerAPI(http_server_uri="http://localhost:60611")
RM.set_authorization_key(api_key="secret")

In [ ]:
ENERGIES = list(np.linspace(6100, 7900, 10))
SI_MATERIAL = "si111"
IMAGE_THRESHOLD = 0.2
IMAGE_BLUR = 2.0

In [ ]:
# Determine target centroid
plan = BPlan(
    "acquire_with_energy_scan",
    [{'m2-yaw': 0.0, 'm2-center_x': 0.0, 'dcm_c2-roll': 0.0}],
    ['m2.yaw', 'm2.center_x', 'dcm_c2.roll'],
    ["xas_det"],
    tpw="tpw",
    dcm_c1="dcm_c1", 
    dcm_c2="dcm_c2",
    crystal=SI_MATERIAL,
    energies=[7112], # Only one energy for baseline
)
RM.item_add(plan)

In [ ]:
image = tiled_client["1f8560f0-d47f-4fc0-afce-fab25478ef20"]["primary/xas_det_image"].read().squeeze()
res = analyze_image(image, threshold=IMAGE_THRESHOLD, blur=IMAGE_BLUR)
target_centroid = (res["x_centroid"], res["y_centroid"])
res

In [ ]:
agent = build_qs_agent(
    RM,
    ["xas_det"],
    "si111",
    tiled_client,
    "xas_det_image",
    target_centroid=target_centroid,
    energies=ENERGIES,
    threshold=IMAGE_THRESHOLD,
    blur=IMAGE_BLUR,
    checkpoint_path="/tmp/blop/energy-alignment.json",
)

In [ ]:
fut = agent.run(iterations=100)

In [ ]:
fut.result(timeout=600)

In [ ]:
agent.ax_client.compute_analyses()